This notebook builds the files of the [search development plan](index.qmd) that follow by fixed rules from the files in `data/raw/search-development/`, writes them to `data/processed/search-development/`, and shows the outcomes of the plan's three steps. Step 1 compares the sources @chawinga2019 included with the RDS-Review criteria. Step 2 lists the terms those sources use for each part of the PCC table. Step 3 tests queries built from those terms in PubMed, Scopus and Web of Science.

`summary_of_outcomes.md` states the outcomes in prose, and `log.md` records the judgments behind the work. The checks come first. If any check finds an error, the notebook stops.

In [ ]:
import math
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "_quarto.yml").exists())
RAW = ROOT / "data" / "raw" / "search-development"
PROCESSED = ROOT / "data" / "processed" / "search-development"
PROCESSED_FILES = {"reference_sources.csv", "counts.csv", "verdict.csv", "statement_matches.csv",
                   "terms.csv", "term_syntax.csv"}
DATABASES = {"PubMed": "pubmed", "Scopus": "scopus", "Web of Science": "web-of-science"}
PARTS = ["sharing", "reuse", "factors that obstruct", "factors that facilitate", "stakeholder perspectives",
         "policies and what drive them", "mechanisms for implementation"]


def path(name):
    return (PROCESSED if name in PROCESSED_FILES else RAW) / name


def read(name):
    return pd.read_csv(path(name), dtype=str, keep_default_na=False)


## Build

Step 1's derived files, `reference_sources.csv`, `counts.csv`, `verdict.csv` and `statement_matches.csv`, from `reference_candidates.csv`, `reference_lookups.csv` and the codes of record. Set, `tier`, Reason, In scope and Eligible apart from year follow the rules in the plan.

Step 2's `terms.csv` and `term_syntax.csv`, from the codes of record, `reference_indexing.csv` and the assignments of record, by the plan's normalisation, syntax and selection rules. Only development set sources supply terms, and a code with a problem from the coding checks supplies none.

In [ ]:
import re

CODES = "reference_codes_qwen3.5-122b.csv"
THEMES = ["factors that obstruct", "factors that facilitate", "stakeholder perspectives",
          "policies and what drive them", "mechanisms for implementation"]
STATEMENTS = [("research data sharing", "researchers"), ("digital curation", "data sharing"),
              ("research data sharing", "open science"), ("research data sharing", "open data")]

cand = read("reference_candidates.csv")
look = read("reference_lookups.csv")
codes = read(CODES)


def write(df, name):
    PROCESSED.mkdir(parents=True, exist_ok=True)
    df.to_csv(PROCESSED / name, index=False, lineterminator="\n")


# one value per source for the single-value fields
single = (codes[codes["field"].isin(["practice", "data_origin", "field", "source_type", "language"])]
          .pivot(index="ref_id", columns="field", values="value"))
themes_of = codes[(codes["field"] == "factors") & (codes["value"] != "none")].groupby("ref_id")["theme"].agg(set)

src = look[["ref_id", "title", "pub_date", "doi", "pmid", "abstract_source"]].merge(
    cand[["ref_id", "label", "cited_in"]], on="ref_id").join(single, on="ref_id")
src["year"] = src["pub_date"].str[:4]
src["set"] = src["year"].astype(int).map(lambda y: "development" if y <= 2014 else "test")
src["tier"] = src["field"].map(lambda f: "1" if f == "health" else "2")
src["coded_on"] = src["ref_id"].map(codes.groupby("ref_id")["coded_on"].first())
src["themes"] = src["ref_id"].map(lambda r: "; ".join(sorted(themes_of.get(r, set()))))


def reasons(s, scope):
    """The codebook.yml exclusion codes a source meets, in the codebook's order. Year and tier count
    only for In scope, not for Eligible apart from year."""
    out = []
    if s["practice"] == "neither":
        out.append("not-data-sharing")
    if not s["themes"]:
        out.append("not-factors")
    if s["data_origin"] == "not-research-data":
        out.append("not-research-data")
    if scope and s["tier"] == "2" and s["source_type"] != "review":
        out.append("tier2-not-review")
    if s["language"] not in ("English", "French"):
        out.append("language")
    if scope and not 2015 <= int(s["year"]) <= 2025:
        out.append("date")
    return out


src["all_reasons"] = src.apply(lambda s: ";".join(reasons(s, True)), axis=1)
src["reason"] = src["all_reasons"].str.split(";").str[0]
src["in_scope"] = src["all_reasons"].map(lambda r: "no" if r else "yes")
src["eligible_apart_from_year"] = src.apply(lambda s: "no" if reasons(s, False) else "yes", axis=1)
src = src[["ref_id", "label", "title", "year", "pub_date", "doi", "pmid", "cited_in", "set", "field", "tier",
           "source_type", "practice", "data_origin", "themes", "language", "coded_on", "abstract_source",
           "eligible_apart_from_year", "in_scope", "reason", "all_reasons"]].sort_values("ref_id")
write(src, "reference_sources.csv")

# counts: sources with each value of each coded field, by set, and codes resting on no passage
setof = src.set_index("ref_id")["set"]
c = codes.assign(set=codes["ref_id"].map(setof), nopassage=codes["passage"].eq("none") & codes["field"].ne("language"))
rows = []
for (field, value, theme), g in c.groupby(["field", "value", "theme"]):
    per = g.groupby("ref_id").agg(set=("set", "first"), nopassage=("nopassage", "all"))
    rows.append(dict(field=field, value=value, theme=theme,
                     development=int((per["set"] == "development").sum()), test=int((per["set"] == "test").sum()),
                     total=len(per), codes_without_passage=int(g["nopassage"].sum())))
for field in ("tier", "eligible_apart_from_year", "in_scope", "reason"):
    for value, g in src.groupby(field):
        rows.append(dict(field=field, value=value or "(none)", theme="",
                         development=int((g["set"] == "development").sum()), test=int((g["set"] == "test").sum()),
                         total=len(g), codes_without_passage=""))
write(pd.DataFrame(rows), "counts.csv")

# verdict: sources eligible apart from year coded for each PCC part, by set
elig = src[src["eligible_apart_from_year"] == "yes"]
has = {"sharing": elig["practice"].isin(["sharing", "both"]), "reuse": elig["practice"].isin(["reuse", "both"])}
has.update({t: elig["themes"].str.contains(t, regex=False) for t in THEMES})
vrows = []
for part, mask in has.items():
    d, t = int((mask & (elig["set"] == "development")).sum()), int((mask & (elig["set"] == "test")).sum())
    verdict = "covered" if d and t else "development only" if d else "test only" if t else "not covered"
    vrows.append(dict(pcc_part=part, development=d, test=t, verdict=verdict))
write(pd.DataFrame(vrows), "verdict.csv")

# Chawinga and Zinn's search statements against titles and abstracts, as phrases and as separate words
squash = lambda s: " " + re.sub(r"[^a-z0-9]+", " ", s.lower()).strip() + " "
text = look.set_index("ref_id").apply(lambda r: squash(r["title"] + " " + r["abstract"]), axis=1)
inscope = set(src.loc[src["in_scope"] == "yes", "ref_id"])
srows = []
for a, b in STATEMENTS:
    for reading in ("phrase", "words"):
        if reading == "phrase":
            hit = lambda t: all(squash(x) in t for x in (a, b))
        else:
            hit = lambda t: all(f" {w} " in t for x in (a, b) for w in squash(x).split())
        ids = sorted(r for r, t in text.items() if hit(t))
        srows.append(dict(statement=f"{a} AND {b}", reading=reading, matched=len(ids),
                          matched_in_scope=sum(r in inscope for r in ids), ref_ids=";".join(ids)))
write(pd.DataFrame(srows), "statement_matches.csv")
print(f"Built reference_sources.csv ({len(src)} sources), counts.csv, verdict.csv and statement_matches.csv.")

# Step 2's terms: recorded terms from the codes, and MeSH headings, author keywords and OpenAlex keywords
# placed under a PCC part by the assignments of record.
ASSIGNMENTS = "term_assignments_qwen3.5-122b.csv"
DBS = ["PubMed", "Scopus", "Web of Science"]
FIELD = {"PubMed": "[tiab]", "Scopus": "TITLE-ABS-KEY", "Web of Science": "TS"}
# Fallback terms from the Candidate terms table on the search strategy page, for a Practice value with no
# development set source, or whose development set sources all carry the other value too.
FALLBACK = {"reuse": ["data reuse", "data re-use"]}

dev = src[src["set"] == "development"]
dev_ids = set(dev["ref_id"])
doi_of = src.set_index("ref_id")["doi"]
assign = read(ASSIGNMENTS)
indexing = read("reference_indexing.csv")


def norm(t):
    t = t.lower().strip().strip('"').replace("’", "'")
    return re.sub(r"\s+", " ", re.sub(r"[-‑]", " ", t)).strip()


def stem(w):
    for suffix in ("ies", "es", "s"):
        if w.endswith(suffix) and len(w) > 4:
            return w[:-len(suffix)] + ("y" if suffix == "ies" else "")
    return w


def merge_key(t):
    """Variants that differ only by hyphen, plural or word ending share a key."""
    return " ".join(stem(w) for w in norm(t).split())


def parts_of(value, theme):
    if value in ("none", "neither"):
        return []
    if value == "both":
        return ["sharing", "reuse"]
    if value in ("sharing", "reuse"):
        return [value]
    return [theme] if theme else []


# occurrences: (key, type, part) -> sources carrying it, its variants, and its assignment basis
occ = {}


def add(key, kind, part, ref_id, variant, basis):
    o = occ.setdefault((key, kind, part), {"src": set(), "variants": set(), "basis": basis})
    o["src"].add(ref_id)
    o["variants"].add(variant)


usable = codes[codes["ref_id"].isin(dev_ids) & codes["field"].isin(["practice", "factors"])
               & ~codes["term"].isin(["", "none"]) & (codes["problem"] == "")]
for _, c in usable.iterrows():
    for part in parts_of(c["value"], c["theme"]):
        add(merge_key(c["term"]), "recorded", part, c["ref_id"], norm(c["term"]), "recorded")
value_of = {(a["type"], a["term"]): (a["value"], a["theme"]) for _, a in assign.iterrows()}
for _, i in indexing[indexing["ref_id"].isin(dev_ids)].iterrows():
    value, theme = value_of.get((i["type"], i["term"]), ("none", ""))
    for part in parts_of(value, theme):
        key = i["term"] if i["type"] == "MeSH" else merge_key(i["term"])
        add(key, i["type"], part, i["ref_id"], i["term"] if i["type"] == "MeSH" else norm(i["term"]), value)

# coded sources per PCC part, and fallback terms where a Practice value lacks its own sources
coded = {p: set() for p in PARTS}
for _, c in codes[codes["ref_id"].isin(dev_ids)].iterrows():
    if c["field"] in ("practice", "factors"):
        for part in parts_of(c["value"], c["theme"]):
            coded[part].add(c["ref_id"])
practice = dev.set_index("ref_id")["practice"]
for part, terms_ in FALLBACK.items():
    own = {r for r in coded[part] if practice[r] == part}
    if not own:
        for t in terms_:
            occ[(merge_key(t), "fallback", part)] = {"src": set(), "variants": {norm(t)}, "basis": "Candidate terms table"}


def written(key, variants, kind, db):
    """One database's written form: MeSH as [mh:noexp] in PubMed only; single words sharing a stem of
    5 or more letters as truncation; otherwise the variants, phrases in quotes, joined with OR."""
    if kind == "MeSH":
        return f'"{key}"[mh:noexp]' if db == "PubMed" else None
    vs = sorted(variants)
    if all(" " not in v for v in vs) and (len(vs) > 1 or stem(vs[0]) != vs[0]):
        root = min((stem(v) for v in vs), key=len)
        while not all(v.startswith(root) for v in vs):
            root = root[:-1]
        forms = [root + "*"] if len(root) >= 5 else vs
    else:
        forms = [f'"{v}"' if " " in v else v for v in vs]
    if db == "PubMed":
        return " OR ".join(f"{f}[tiab]" for f in forms)
    q = " OR ".join(forms)
    return f"TITLE-ABS-KEY({q})" if db == "Scopus" else f"TS=({q})"


shown = lambda key, o, kind: key if kind == "MeSH" else min(o["variants"], key=lambda v: (len(v), v))
syntax_rows, term_rows = [], []
for db in DBS:
    here = {k: o for k, o in occ.items() if not (k[1] == "MeSH" and db != "PubMed")}
    seen = set()
    for (key, kind, part), o in sorted(here.items()):
        if (key, kind) not in seen:
            seen.add((key, kind))
            syntax_rows.append(dict(term=shown(key, o, kind), type=kind, database=db,
                                    variants="; ".join(sorted(o["variants"])),
                                    written_form=written(key, o["variants"], kind, db)))
    for part in PARTS:
        ranked = sorted(((k, o) for k, o in here.items() if k[2] == part),
                        key=lambda x: (-len(x[1]["src"]), x[0][0], x[0][1]))
        matched = set()
        for (key, kind, _), o in ranked:
            row = dict(term=shown(key, o, kind), type=kind, database=db,
                       field="[mh:noexp]" if kind == "MeSH" else FIELD[db], pcc_part=part,
                       sources_in_part=len(coded[part]), assignment_basis=o["basis"], count=len(o["src"]))
            if part in ("sharing", "reuse"):
                row.update(sources_added="", dois=";".join(doi_of[r] or r for r in sorted(o["src"])), status="")
            else:
                added = o["src"] - matched
                if not added:
                    continue
                matched |= added
                row.update(sources_added=len(added), dois=";".join(doi_of[r] or r for r in sorted(added)),
                           status="to screening")
            term_rows.append(row)
write(pd.DataFrame(syntax_rows), "term_syntax.csv")
write(pd.DataFrame(term_rows), "terms.csv")
print(f"Built term_syntax.csv ({len(syntax_rows)} written forms) and terms.csv ({len(term_rows)} rows).")


In [ ]:
steps_1_2 = ["reference_candidates.csv", "reference_sources.csv", CODES, "counts.csv",
             "verdict.csv", "comparison.csv", "statement_matches.csv", "terms.csv", "term_syntax.csv"]
missing = [n for n in steps_1_2 if not path(n).exists()]
have_data = not missing
if not have_data:
    print("Not run yet. Missing: " + ", ".join(missing))
else:
    cand, src, codes, counts, verdict, comparison, statements, terms, syntax = (read(n) for n in steps_1_2)
have_tests = path("query_tests.csv").exists()

## Checks

In [ ]:
if have_data:
    errors = []
    ids = set(src["ref_id"])
    if set(cand["ref_id"]) != ids:
        errors.append(f"reference_sources.csv and reference_candidates.csv list different sources: {sorted(set(cand['ref_id']) ^ ids)}.")
    unknown = set(codes["ref_id"]) - ids
    if unknown:
        errors.append(f"{CODES} has sources not in reference_sources.csv: {sorted(unknown)}.")
    for f in ("practice", "data_origin", "field", "source_type", "language"):
        n = codes[codes["field"] == f].groupby("ref_id").size().reindex(sorted(ids), fill_value=0)
        if (n != 1).any():
            errors.append(f"{int((n != 1).sum())} source(s) lack exactly one {f} code: {sorted(n[n != 1].index)}.")
    pairs = set(zip(syntax["term"], syntax["type"], syntax["database"]))
    unwritten = [t for t in zip(terms["term"], terms["type"], terms["database"]) if t not in pairs]
    if unwritten:
        errors.append(f"{len(unwritten)} row(s) in terms.csv have no written form in term_syntax.csv: {unwritten[:5]}.")
    if errors:
        raise ValueError("\n".join(errors))
    print("All checks passed.")
    flagged = codes[codes["problem"] != ""]
    if len(flagged):
        print(f"{len(flagged)} code(s) in {CODES} carry a problem from the coding checks:")
        display(flagged[["ref_id", "field", "value", "problem", "passage"]].reset_index(drop=True))


## Step 1. Comparability assessment

The reference set holds the references @chawinga2019 name in Table 2, cite in sections 4 and 5, or cite for a country section 4.3 lists. A local model coded each one from its title and abstract. Sources first published from 2007 to 2014 form the development set, which step 2 draws terms from. Sources first published from 2015 to July 2018 form the test set, which step 3 measures coverage against.

In [ ]:
if have_data:
    sets = pd.crosstab(src["set"], src["eligible_apart_from_year"], margins=True, margins_name="total")
    sets.columns.name = "eligible apart from year"
    display(sets)
    reasons = (src.assign(reason=src["reason"].replace("", "(none)"))
               .groupby(["reason", "set"]).size().unstack(fill_value=0))
    display(reasons)

### Codes

Each count is the number of sources with that value. `codes_without_passage` counts the codes no sentence of the title or abstract supports.

In [ ]:
if have_data:
    coded = ["practice", "data_origin", "factors", "field", "source_type", "language"]
    display(counts[counts["field"].isin(coded)].reset_index(drop=True))
    display(counts[~counts["field"].isin(coded)].reset_index(drop=True))


### Criteria compared

In [ ]:
if have_data:
    n = len(src)
    by = lambda col: ", ".join(f"{v} {c}" for v, c in src[col].value_counts().items())
    evidence = {
        "source type": by("source_type"),
        "empirical design": f"empirical {int((src['source_type'] == 'empirical').sum())} of {n}",
        "language": by("language"),
        "date": f"2007-2014 {int((src['set'] == 'development').sum())}, 2015-2018 {int((src['set'] == 'test').sum())}",
        "topic": f"practice: {by('practice')}",
        "reuse": f"practice: {by('practice')}",
        "data_origin": by("data_origin"),
        "factors": f"sources with no factor: {int((src['themes'] == '').sum())} of {n}",
        "field and tier": by("field"),
    }
    display(comparison.assign(evidence=comparison["criterion"].map(evidence)))


### Search statements

The four search statements @chawinga2019 report, matched against the titles and abstracts of the reference set, once with each part as a phrase and once as separate words.

In [ ]:
if have_data:
    display(statements.drop(columns=["ref_ids"]))

### Verdict

A PCC part is *covered* when sources eligible apart from year carry it in both sets.

In [ ]:
if have_data:
    display(verdict)

## Step 2. Terms by PCC part

Terms come from the development set: the terms recorded in the coding, and MeSH headings, author keywords and OpenAlex keywords the local model placed under a PCC part. *Sharing* and *reuse* terms go to step 3. Factor terms go to screening. A term from a source coded *both* sits under *sharing* and *reuse*.

In [ ]:
if have_data:
    n = terms.groupby(["pcc_part", "database"]).size().unstack(fill_value=0).reindex(PARTS)
    display(n)

In [ ]:
if have_data:
    sr = terms[(terms["database"] == "PubMed") & terms["pcc_part"].isin(["sharing", "reuse"])].copy()
    sr["count"] = sr["count"].astype(int)
    display(sr.sort_values(["pcc_part", "count"], ascending=[True, False])[["pcc_part", "term", "type", "count"]].reset_index(drop=True))

In [ ]:
if have_data:
    fac = terms[(terms["database"] == "PubMed") & ~terms["pcc_part"].isin(["sharing", "reuse"])]
    display(fac[["pcc_part", "term", "type", "count", "sources_added"]].reset_index(drop=True))

## Step 3. Query tests

Each database gets one query, built by the yield rule in the plan and tested for coverage of the test set. Precision comes from a random sample of 100 records per database, screened on title and abstract.

In [ ]:
def wilson(k, n, z=1.96):
    if n == 0:
        return float("nan"), float("nan"), float("nan")
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return round(p, 3), round(max(0, c - h), 3), round(min(1, c + h), 3)


if not have_tests:
    print("Not run yet. Missing: query_tests.csv")
else:
    tests = read("query_tests.csv")
    display(tests.set_index("database").drop(columns=["final_query"]).T)
    for _, t in tests.iterrows():
        print(f"{t['database']}: {t['final_query']}")

In [ ]:
if have_tests:
    if not path("term_results.csv").exists():
        print("Not run yet. Missing: term_results.csv")
    else:
        res = read("term_results.csv")
        display(res.groupby(["database", "status"]).size().unstack(fill_value=0))
        inq = res[res["status"].isin(["in query", "fallback in query"])]
        display(inq[["database", "term", "type", "development_yield", "load_2015_2025", "sources_added"]].reset_index(drop=True))


### Misses

Indexed test set sources the query misses, and development set sources reached only by terms excluded by yield.

In [ ]:
if have_tests and path("query_misses.csv").exists():
    display(read("query_misses.csv"))

### Precision

Precision counts *undecidable* records once as included and once as left out.

In [ ]:
if have_tests:
    rows = []
    for db, slug in DATABASES.items():
        p = path(f"precision_sample_{slug}.csv")
        if not p.exists():
            continue
        s = read(p.name)
        d = s["decision"].str.lower()
        if (d == "").any():
            print(f"{db}: {int((d == '').sum())} of {len(s)} sampled records not screened yet.")
            continue
        inc, exc, und = int((d == "include").sum()), int((d == "exclude").sum()), int((d == "undecidable").sum())
        a, b = wilson(inc + und, len(s)), wilson(inc, inc + exc)
        rows.append(dict(database=db, sampled=len(s), included=inc, excluded=exc, undecidable=und,
                         title_only=int((s["abstract_source"] == "none").sum()),
                         precision_undecidable_included=a[0], ci_included=f"{a[1]}-{a[2]}",
                         precision_undecidable_left_out=b[0], ci_left_out=f"{b[1]}-{b[2]}"))
    if rows:
        display(pd.DataFrame(rows).set_index("database"))